### Multi Agent-Systems
A team is a group of agents that work together to achieve a common goal

In [1]:
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage

from dotenv import load_dotenv
import os

In [2]:
from azure.keyvault.secrets import SecretClient
from azure.identity import DefaultAzureCredential


load_dotenv()

vault_url = os.getenv("AZURE_VAULT_URL")

credential = DefaultAzureCredential()

client = SecretClient(credential=credential, vault_url=vault_url)

open_api_key = client.get_secret("openai-api-key").value
open_api_version = client.get_secret("openai-api-version").value
openai_deployment = client.get_secret("openai-deployment").value
openai_endpoint = client.get_secret("openai-endpoint").value

In [ ]:
model_client = AzureOpenAIChatCompletionClient(
    model="gpt-4o-2024-11-20",
    azure_deployment=openai_deployment,
    api_key=open_api_key,
    api_version=open_api_version,
    azure_endpoint=openai_endpoint,
)

### Single Agent Approach
A single agent is going to create a short story for us

In [4]:
story_agent = AssistantAgent(
    name = 'story_agent',
    model_client=model_client,
    system_message='You are a creating writer. Generate a short story about a brave knight and a dragon.'
)

In [5]:
async def test_simple_agent():
    task = TextMessage(
        content='Write a short story about a brave knight and a dragon. Keep it up to 50 words',
        source='user'
    )
    result = await story_agent.run(task=task)
    print(result.messages[-1].content)

await test_simple_agent()

Sir Alden faced the blazing dragon, heart steady. "Bold beast, I seek peace!" he declared. The dragon paused, intrigued. Alden offered his bread, sharing unseen struggles. The mighty creature roared and flew away, sparing the kingdom. Bravery, Alden learned, wasn't in blade alone—but in daring to hold out hope.


d:\All Python\Microsoft Autogen\.venv\Lib\site-packages\autogen_agentchat\agents\_assistant_agent.py:1109: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


### Multi Agent Team Approach
We will be using 3 agents to now write the story, each with its own work and mastery

In [9]:
plot_agent = AssistantAgent(
    name = 'plot_writer',
    model_client=model_client,
    system_message="You create engaging plots for stories. Focus on the Pokemon's journey"
)

character_agent = AssistantAgent(
    name = 'character_writer',
    model_client=model_client,
    system_message="You develop characters. Describe the pokemon and the villian in detail, including their motivations and backgrounds."
)

ending_agent = AssistantAgent(
    name = 'ending_writer',
    model_client=model_client,
    system_message="You write engaging endings. conclude the story with a twist."
)

RoundRobinGroupChat is a simple yet effective team configuration where all agents share the same context and take turns responding in a round-robin fashion. Each agent, during its turn, broadcasts its response to all other agents, ensuring that the entire team maintains a consistent context.

In [7]:
from autogen_agentchat.teams import RoundRobinGroupChat

team = RoundRobinGroupChat(
    participants=[plot_agent,character_agent,ending_agent],
    max_turns=6
)
team.component_type

'team'

In [8]:
async def test_team():
    task = TextMessage(
        content='Write a short story a brave boy and his Pokemon. Keep it up to 50 words',
        source='user'
    )

    result = await team.run(task=task)
    for each_agent_message in result.messages:
        print(f'{each_agent_message.source} : {each_agent_message.content}')

await test_team()

d:\All Python\Microsoft Autogen\.venv\Lib\site-packages\autogen_agentchat\agents\_assistant_agent.py:1109: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


user : Write a short story a brave boy and his Pokemon. Keep it up to 50 words
plot_writer : In a dense forest, brave Luka and his loyal Charmander faced a wild Ursaring. "Flamethrower, now!" Luka commanded. Fire blazed, creating an escape route. Charmander's tail lit their path forward. Exhausted but unbeaten, they reached the mountaintop at sunrise, their bond forged stronger, ready for their next adventure.
character_writer : Would you like me to expand on this story or add more depth to the characters and their journey?
ending_writer : Sure! Let’s add depth and a twist. Here's the ending with sparkle and surprise:

Exhausted but unbeaten, they reached the mountaintop at sunrise. Luka smiled as Charmander’s tail flickered; they believed victory lay ahead. But as sunlight touched the peak, it transformed, revealing a hidden shrine. Inside, ancient carvings awakened—a prophecy about Luka and Charmander’s destined battle… against each other!
plot_writer : Luka’s heart sank as he read t